####Урок 10. Машинный перевод. Модель seq2seq и механизм внимания.
**Задание.**

Разобраться с моделью перевода, как она устроена, запустить для перевода с русского на английский (при желании можно взять другие пары языков).


In [1]:
# Подключаем диск
from google.colab import drive
drive.mount('./drive')

Mounted at ./drive


In [2]:
import tensorflow as tf

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.model_selection import train_test_split

import unicodedata
import re
import numpy as np
import os
import io
import time

**Загрузка данных.**

We'll use a language dataset provided by http://www.manythings.org/anki/

In [3]:
#Скачиваем датасет
!wget http://www.manythings.org/anki/rus-eng.zip

--2023-12-17 18:43:24--  http://www.manythings.org/anki/rus-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16065699 (15M) [application/zip]
Saving to: ‘rus-eng.zip’

rus-eng.zip         100%[===================>]  15.32M  18.6MB/s    in 0.8s    

2023-12-17 18:43:25 (18.6 MB/s) - ‘rus-eng.zip’ saved [16065699/16065699]



In [4]:
#Создаем папку и извлекаем туда файлы из архива
!mkdir rus-eng
!unzip rus-eng.zip -d rus-eng/

Archive:  rus-eng.zip
  inflating: rus-eng/rus.txt         
  inflating: rus-eng/_about.txt      


In [5]:
!ls /content/rus-eng/ -lah

total 78M
drwxr-xr-x 2 root root 4.0K Dec 17 18:43 .
drwxr-xr-x 1 root root 4.0K Dec 17 18:43 ..
-rw-r--r-- 1 root root 1.5K Nov 29 00:30 _about.txt
-rw-r--r-- 1 root root  78M Nov 29 00:30 rus.txt


In [3]:
# Загружаем файл
path_to_file = '/content/drive/MyDrive/NLP/rus-eng/rus.txt'

In [4]:
#просмотр файла
f = open(path_to_file)
for line in f:
    print(line)
    break

Go.	Марш!	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1159202 (shanghainese)



**Подготовка данных.**

In [5]:
#функция препроцессинга
def preprocess_sentence(w):
  #переводим предложение к нижнему регистру и удалем начальные и конечные пробелы
    w = w.lower().strip()

  # отделяем пробелом слово и следующую за ним пунктуацию
  # пример: "he is a boy." => "he is a boy ."
    w = re.sub(r"([?.!,])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

  # все, кроме букв и знаков пунктуации, заменяем пробелом
    w = re.sub(r"[^a-zA-Zа-яА-Я?.!,']+", " ", w)

  #удаляем лишние пробелы в начале и конце
    w = w.strip()

  # создаем начало и конец последовательности
  # теперь модель знает, где начинать и заканчивать предсказания
    w = '<start> ' + w + ' <end>'
    return w

In [6]:
#пример работы препроцессинга
preprocess_sentence("I can't go.")

"<start> i can't go . <end>"

In [7]:
# 1. Убираем акценты
# 2. Очищаем предложения
# 3. Возвращаем пары слов: [ENG, RUS]
def create_dataset(path, num_examples):
  #считываем строки файла
    lines = io.open(path, encoding='UTF-8').read().strip().split('\n')
  #каждую строку разделяем на пробелы, берем первые 2 слова, препроцессим их и возвращаем пару
    word_pairs = [[preprocess_sentence(w) for w in l.split('\t')[:2]]  for l in lines[:num_examples]]

    return zip(*word_pairs)

In [8]:
#пример работы
en, ru = create_dataset(path_to_file, None)
print(en[0])
print(ru[0])

<start> go . <end>
<start> марш ! <end>


In [9]:
# количество данных в датасете
len(en), len(ru)

(487600, 487600)

In [10]:
def tokenize(lang):
      #токенизируем текст, отфильтвовываем пробелы
    lang_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
      #обновляем внутренний словарь на основе lang
    lang_tokenizer.fit_on_texts(lang)
      #преобразуем каждый элемент из lang в последовательность чисел
    tensor = lang_tokenizer.texts_to_sequences(lang)
      #преобразуем тензор в матрицу (кол-во тензоров * max-длина),
      #при этом короткие последовательности заполняем нулями сзади, а длинные -- обрезаем сзади
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')
    return tensor, lang_tokenizer

**Создание датасета.**

In [11]:
def load_dataset(path, num_examples=None):
      # создаем очищенные анг (выходные), русские (входные) пары
    targ_lang, inp_lang = create_dataset(path, num_examples)
    #применяем токенизацию к каждому элементы из пары
    input_tensor, inp_lang_tokenizer = tokenize(inp_lang)
    target_tensor, targ_lang_tokenizer = tokenize(targ_lang)

    return input_tensor, target_tensor, inp_lang_tokenizer, targ_lang_tokenizer

In [12]:
num_examples = 100000
input_tensor, target_tensor, inp_lang, targ_lang = load_dataset(path_to_file, num_examples)

input_tensor, target_tensor

(array([[    1,  5701,    21, ...,     0,     0,     0],
        [    1,   182,     3, ...,     0,     0,     0],
        [    1,   277,     3, ...,     0,     0,     0],
        ...,
        [    1,     7,  1108, ...,     0,     0,     0],
        [    1,     7, 20519, ...,     0,     0,     0],
        [    1,    25,  1108, ...,     0,     0,     0]], dtype=int32),
 array([[ 1, 27,  3, ...,  0,  0,  0],
        [ 1, 27,  3, ...,  0,  0,  0],
        [ 1, 27,  3, ...,  0,  0,  0],
        ...,
        [ 1,  5,  8, ...,  0,  0,  0],
        [ 1,  5,  8, ...,  0,  0,  0],
        [ 1,  5,  8, ...,  0,  0,  0]], dtype=int32))

In [13]:
# Вычисляем максимальную длину тензоров
max_length_targ, max_length_inp = target_tensor.shape[1], input_tensor.shape[1]
max_length_targ, max_length_inp

(11, 15)

In [14]:
# Создаем тренировочные и валидационные датасеты
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

# размеры датасетов
print(len(input_tensor_train), len(target_tensor_train), len(input_tensor_val), len(target_tensor_val))

80000 80000 20000 20000


In [15]:
input_tensor_train, target_tensor_train

(array([[    1,    12,  2798, ...,     0,     0,     0],
        [    1,     4,  6145, ...,     0,     0,     0],
        [    1,     7,   784, ...,     0,     0,     0],
        ...,
        [    1,     4, 10609, ...,     0,     0,     0],
        [    1,     8,     6, ...,     0,     0,     0],
        [    1,     4,  1351, ...,     0,     0,     0]], dtype=int32),
 array([[   1,   22,    7, ...,    0,    0,    0],
        [   1,   13,   92, ...,    0,    0,    0],
        [   1,    5,  523, ...,    0,    0,    0],
        ...,
        [   1,    4, 1969, ...,    0,    0,    0],
        [   1,   28,   31, ...,    0,    0,    0],
        [   1,    4,  947, ...,    0,    0,    0]], dtype=int32))

In [16]:
#функция получения из токена текста (выводим токен и его индекс)
def convert(lang, tensor):
    for t in tensor:
        if t!=0:
            print ("%d ----> %s" % (t, lang.index_word[t]))

In [17]:
print ("Input Language; index to word mapping")
convert(inp_lang, input_tensor_train[0])
print ()
print ("Target Language; index to word mapping")
convert(targ_lang, target_tensor_train[0])

Input Language; index to word mapping
1 ----> <start>
12 ----> вы
2798 ----> полицейский
5 ----> ?
2 ----> <end>

Target Language; index to word mapping
1 ----> <start>
22 ----> are
7 ----> you
9 ----> a
2300 ----> policeman
6 ----> ?
2 ----> <end>


In [18]:
BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64
#количество эпох
steps_per_epoch = len(input_tensor_train)//BATCH_SIZE
embedding_dim = 300
units = 1024
vocab_inp_size = len(inp_lang.word_index)+1
vocab_tar_size = len(targ_lang.word_index)+1
#из каждого элемента (input_tensor_train, target_tensor_train) создает тензор
dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
#разбиваем датасет на батчи (списки по 64), удаляя последний неполный батч
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

In [19]:
example_input_batch, example_target_batch = next(iter(dataset))
print(example_input_batch.shape, example_target_batch.shape)
example_input_batch[0], example_target_batch[0]

(64, 15) (64, 11)


(<tf.Tensor: shape=(15,), dtype=int32, numpy=
 array([    1, 10657,     3,     2,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0], dtype=int32)>,
 <tf.Tensor: shape=(11,), dtype=int32, numpy=array([  1,  28, 459, 193,   3,   2,   0,   0,   0,   0,   0], dtype=int32)>)

**Построение модели машинного перевода.**

Encoder

In [24]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units,
                                       return_sequences=False,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')
    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state = hidden)
        return state

    def initialize_hidden_state(self):
    #создаем тензор из нулей размера (батч, кол-во ячеек)
        return tf.zeros((self.batch_sz, self.enc_units))

In [25]:
encoder = Encoder(vocab_inp_size, embedding_dim, units, BATCH_SIZE)

# инициализитеруем начальное скрытое состояние (из нулей)
sample_hidden = encoder.initialize_hidden_state()
# применяем энкодер к входному батчу и скрытому состоянию
sample_hidden = encoder(example_input_batch, sample_hidden)
# print ('Форма выхода энкодера: (batch size, sequence length, units) {}'.format(sample_output.shape))
print ('Encoder Hidden state shape: (batch size, units) {}'.format(sample_hidden.shape))

Encoder Hidden state shape: (batch size, units) (64, 1024)


In [26]:
sample_hidden

<tf.Tensor: shape=(64, 1024), dtype=float32, numpy=
array([[-0.00017617,  0.00278847, -0.01035333, ..., -0.00463423,
         0.00899644,  0.01043913],
       [ 0.00056296,  0.00303296, -0.01075401, ..., -0.0050865 ,
         0.00904993,  0.01043557],
       [ 0.00047836,  0.00305762, -0.01073896, ..., -0.00497222,
         0.00917403,  0.01047888],
       ...,
       [ 0.00048723,  0.00306988, -0.01081371, ..., -0.00507343,
         0.00904974,  0.01038944],
       [ 0.00061089,  0.00298613, -0.01075106, ..., -0.00504112,
         0.00906038,  0.01051992],
       [ 0.00055455,  0.00299596, -0.0107854 , ..., -0.00508341,
         0.00911994,  0.0104334 ]], dtype=float32)>

Decoder

In [27]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.dec_units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, hidden):
        # x shape после прохождения через эмбеддинг == (batch_size, 1, embedding_dim)
        x = self.embedding(x)

        # отправляем в GRU входные данные и скрытое состояние (от энкодера)
        #выход GRU (batch_size, timesteps, units)
        #размер возвращаемого внутреннего состояния (batch_size, units)
        output, state = self.gru(x, initial_state=hidden)

        # output shape == (batch_size * 1, hidden_size)
        output = tf.reshape(output, (-1, output.shape[2]))

        # x shape == (batch_size, vocab)
        x = self.fc(output)

        return x, state

In [28]:
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)
#применяем декодер к случайному батчу из равномерного распределения (батч,1) и выходу энкодера
decoder_sample_x, decoder_sample_h = decoder(tf.random.uniform((BATCH_SIZE, 1)), sample_hidden)
decoder_sample_x.shape, decoder_sample_h.shape

(TensorShape([64, 7356]), TensorShape([64, 1024]))

In [29]:
decoder_sample_x

<tf.Tensor: shape=(64, 7356), dtype=float32, numpy=
array([[-0.00145532,  0.00142032, -0.00062965, ..., -0.00344973,
         0.00847182, -0.00452251],
       [-0.00148394,  0.00146054, -0.00047392, ..., -0.00351645,
         0.00859623, -0.00434375],
       [-0.0015068 ,  0.00144778, -0.0004873 , ..., -0.00352619,
         0.00858853, -0.00432204],
       ...,
       [-0.00152602,  0.0014376 , -0.00048885, ..., -0.00353318,
         0.00858827, -0.00434022],
       [-0.00149839,  0.00143146, -0.00048961, ..., -0.00350375,
         0.0085788 , -0.00435979],
       [-0.00150295,  0.00145189, -0.00049437, ..., -0.00352211,
         0.00858817, -0.00433029]], dtype=float32)>

Компиляция модели

In [20]:
#оптимизатор
optimizer = tf.keras.optimizers.Adam()

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

#функция потерь
def loss_function(real, pred):
      #делаем инверсию значений сравнения каждого из real с нулем (возвращается true или false)
    mask = tf.math.logical_not(tf.math.equal(real, 0))
      #применяем функцию ошибок к реальным данным и предсказанным
    loss_ = loss_object(real, pred)
      #приводим тензор mask к новому типу loss_.dtype
    mask = tf.cast(mask, dtype=loss_.dtype)
      #умножаем loss_ на mask
    loss_ *= mask
      # возвращаем среднее значениe всех элементов
    return tf.reduce_mean(loss_)

Сheckpoint

In [31]:
checkpoint_dir = './training_nmt_checkpoints'

checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")

checkpoint = tf.train.Checkpoint(optimizer=optimizer,
                                 encoder=encoder,
                                 decoder=decoder)

Обучение модели

In [32]:
@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0
  # перечисляем операции для автоматического дифференцирования
    with tf.GradientTape() as tape:
        #получаем выход encoder
        enc_hidden = encoder(inp, enc_hidden)
        #помещаем его в скрытое состояние decoder
        dec_hidden = enc_hidden
        #формируем вход декодера:
                 # берем список длины батч из индексов тега <start> (1)
                 # приписываем списку размерность 1 сзади (батч, 1)
        dec_input = tf.expand_dims([targ_lang.word_index['<start>']] * BATCH_SIZE, 1)

        # Teacher forcing - выводим target в качестве следующего входа
        for t in range(1, targ.shape[1]):
          # помещаем enc_output в decoder
            predictions, dec_hidden = decoder(dec_input, dec_hidden)
          # считаем функцию потерь
            loss += loss_function(targ[:, t], predictions)
          # используем teacher forcing (приписываем списку размерность 1 сзади)
          #посылаем dec_input на вход декордера
            dec_input = tf.expand_dims(targ[:, t], 1)

    batch_loss = (loss / int(targ.shape[1]))
    #переменные
    variables = encoder.trainable_variables + decoder.trainable_variables
    #вычисляем градиенты loss по variables
    gradients = tape.gradient(loss, variables)
    #оптимизатор применяет подсчитанные градиенты
    optimizer.apply_gradients(zip(gradients, variables))

    return batch_loss

In [33]:
EPOCHS = 50

for epoch in range(EPOCHS):
    start = time.time()

  #инициализируем входное скрытое состояние (из нулей) размера (батч, кол-во рекуррентных ячеек)
    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0

    for (batch, (inp, targ)) in enumerate(dataset.take(steps_per_epoch)):
        #делаем шаг обучения. находим оштбку за этоху
        batch_loss = train_step(inp, targ, enc_hidden)
        #считаем ошибку
        total_loss += batch_loss

        if batch % 100 == 0:
            print('Epoch {} Batch {} Loss {:.4f}'.format(epoch + 1,
                                                       batch,
                                                       batch_loss.numpy()))
  # saving (checkpoint) the model every 2 epochs
    if (epoch + 1) % 2 == 0:
        checkpoint.save(file_prefix = checkpoint_prefix)

    print('Epoch {} Loss {:.4f}'.format(epoch + 1,
                                      total_loss / steps_per_epoch))
    print('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

Epoch 1 Batch 0 Loss 4.5907
Epoch 1 Batch 100 Loss 2.0007
Epoch 1 Batch 200 Loss 1.7730
Epoch 1 Batch 300 Loss 1.6423
Epoch 1 Batch 400 Loss 1.5452
Epoch 1 Batch 500 Loss 1.5998
Epoch 1 Batch 600 Loss 1.4781
Epoch 1 Batch 700 Loss 1.5205
Epoch 1 Batch 800 Loss 1.3795
Epoch 1 Batch 900 Loss 1.1995
Epoch 1 Batch 1000 Loss 1.3025
Epoch 1 Batch 1100 Loss 1.2564
Epoch 1 Batch 1200 Loss 1.2095
Epoch 1 Loss 1.5499
Time taken for 1 epoch 75.91977143287659 sec

Epoch 2 Batch 0 Loss 1.0608
Epoch 2 Batch 100 Loss 1.0860
Epoch 2 Batch 200 Loss 1.0105
Epoch 2 Batch 300 Loss 1.1269
Epoch 2 Batch 400 Loss 1.0251
Epoch 2 Batch 500 Loss 1.0604
Epoch 2 Batch 600 Loss 1.0187
Epoch 2 Batch 700 Loss 0.9139
Epoch 2 Batch 800 Loss 0.9508
Epoch 2 Batch 900 Loss 0.9072
Epoch 2 Batch 1000 Loss 0.9411
Epoch 2 Batch 1100 Loss 0.8912
Epoch 2 Batch 1200 Loss 0.8531
Epoch 2 Loss 0.9950
Time taken for 1 epoch 59.344727754592896 sec

Epoch 3 Batch 0 Loss 0.7952
Epoch 3 Batch 100 Loss 0.7243
Epoch 3 Batch 200 Loss 0.73

Перевод

Функция оценки аналогична циклу обучения, за исключением того, что здесь мы не используем teacher forcing. Входным сигналом для декодера на каждом временном шаге являются его предыдущие предсказания вместе со скрытым состоянием и выходным сигналом энсодера.

Предсказания модели прекращаются, когда модель предскажет end token.

Сохраняем веса внимания для каждого временного шага.

Примечание: Выходной сигнал энкодера вычисляется только один раз для одного входа.

In [34]:
def evaluate(sentence):
  #препоцессим предложение
    sentence = preprocess_sentence(sentence)
      #разбиваем предложение по пробелам и составляем список индексов каждого слова
    inputs = [inp_lang.word_index[i] for i in sentence.split(' ')]
      #добиваем inputs нулями справа до максимальной длины входного текста
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs],
                                                             maxlen=max_length_inp,
                                                             padding='post')
      #преобразуем inputs в тензор
    inputs = tf.convert_to_tensor(inputs)

    result = ''
      # инициализируем входной хидден из нулей размера (1, units)
    hidden = [tf.zeros((1, units))]
      #подаем inputs и hidden в encoder
    enc_hidden = encoder(inputs, hidden)

      #инициализируем входной хидден декодера -- выходной хидден энкодера
    dec_hidden = enc_hidden
      #вход декодера -- список [индекс start] размера(1,1)
    dec_input = tf.expand_dims([targ_lang.word_index['<start>']], 0)

    for t in range(max_length_targ):
            #получаем выход декодера
        predictions, dec_hidden = decoder(dec_input, dec_hidden)

        predicted_id = tf.argmax(predictions[0]).numpy()
        result += targ_lang.index_word[predicted_id] + ' '

   #заканчиваем на токене end
        if targ_lang.index_word[predicted_id] == '<end>':
            return result, sentence

    # предсказанный predicted ID подаем обратно в декодер (размер (1,1))
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, sentence

In [35]:
#функция перевода
def translate(sentence):
    result, sentence = evaluate(sentence)
    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))

In [36]:
# загружаем последний checkpoint in checkpoint_dir
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

In [37]:
translate('Хорошая погода')

Input: <start> хорошая погода <end>
Predicted translation: that's good . <end> 


In [38]:
translate('Я иду в школу')

Input: <start> я иду в школу <end>
Predicted translation: i walk to school . <end> 


In [39]:
translate('Я иду на урок  в школу')

Input: <start> я иду на урок в школу <end>
Predicted translation: i walk to school . <end> 


In [40]:
translate('Я сам решаю трудную задачу в школе')

Input: <start> я сам решаю трудную задачу в школе <end>
Predicted translation: i'm getting old to sing . <end> 


In [41]:
translate(u'Я не люблю, когда идет снег.')

Input: <start> я не люблю , когда идет снег . <end>
Predicted translation: i don't like women . <end> 


In [42]:
translate(u'Сегодня я пойду гулять, а потом пойду в магазин')

Input: <start> сегодня я пойду гулять , а потом пойду в магазин <end>
Predicted translation: i'll go next week . <end> 


In [43]:
translate('Мама приготовила вкусный обед для нашей большой семьи')

Input: <start> мама приготовила вкусный обед для нашей большой семьи <end>
Predicted translation: my mom has arrived . <end> 


In [44]:
translate(u'Мне здесь нравится, но я очень скучаю по дому. Там остались мама и папа.')

Input: <start> мне здесь нравится , но я очень скучаю по дому . там остались мама и папа . <end>
Predicted translation: no , a never get . <end> 
